# MMELON Embedding Extraction

Extract MMELON embeddings for the molecules in a CSV, separately for each view:

| View | Dim | Encoder |
|---|---|---|
| `text` | 768 | SMILES transformer |
| `image` | 512 | ResNet on a 2D drawing of the molecule |
| `graph` | 512 | Graph transformer on the molecular graph |
| `fused` | 512 | Attention-weighted combination of the three (the input to the prediction head), computed per molecule |

**Model:** the pretrained base model (`ibm-research/biomed.sm.mv-te-84m`), or a fine-tuned `best_model.ckpt`
from [MMELON_finetuning.ipynb](MMELON_finetuning.ipynb) or a `fold_<i>/best_model.ckpt` from
[MMELON_finetuning_cv.ipynb](MMELON_finetuning_cv.ipynb). Fine-tuning updates all encoder weights, so a
fine-tuned model gives different embeddings than the base model.

**Environment:** run with the `biomed-multiview` kernel.

**Output** (in the gitignored `aircheck/output/`):
```
output/<DATASET>/embeddings/<EMBEDDING_NAME>/
├── config.json
├── text/part-00000.parquet, part-00001.parquet, ...    # one folder per view
├── image/...
├── graph/...
└── invalid_smiles.csv                                   # only if some SMILES could not be read
```
Each part file has the columns `row_index` (row in the input CSV), `smiles` and `embedding` (a float32 list).
Read a whole view with `pd.read_parquet(".../text")`.

Molecules are processed in chunks of `CHUNK_SIZE`, and each chunk is saved as soon as it is done.
If a long run stops, run the notebook again with the same settings: finished chunks are skipped.

On CPU, extraction takes roughly 0.25 s per molecule; use a GPU for large files.

## 1. Inputs and settings

**Edit these before running.** Data files can be `.csv` or `.parquet`; only the columns the notebook needs are read.

In [1]:
from pathlib import Path

# Dataset name: all outputs go under aircheck/output/<DATASET>/ (the pretrained base model is shared)
DATASET = "sample-data"

# Name of this extraction; outputs go to output/<DATASET>/embeddings/<EMBEDDING_NAME>/
EMBEDDING_NAME = "test-pretrained"

# None = pretrained base model; or a fine-tuned best_model.ckpt (or the folder that contains it)
MODEL_PATH = None

# Molecules
DATA_PATH = Path("~/repos/biomed-multi-view/aircheck/sample_data/test.parquet")
# For the final test set (LightGBM inference): EMBEDDING_NAME = "test-pretrained",
# DATA_PATH = Path("../sample_data/test.parquet")
SMILES_COL = "SMILES"

# Views to save: any of "text", "image", "graph", "fused"
VIEWS = ["text", "image", "graph", "fused"]

In [2]:
import os

BASE_MODEL = "ibm-research/biomed.sm.mv-te-84m"  # used when MODEL_PATH is None

CHUNK_SIZE = 1000         # molecules per saved part file
BATCH_SIZE = 32
NUM_WORKERS = min(24, os.cpu_count())  # data-loading processes: all CPUs, up to 24

In [3]:
import json
import shutil


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "bmfm_sm").is_dir() and (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError(f"Run this notebook from inside the biomed-multi-view repo (cwd={start})")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
OUTPUT_DIR = REPO_ROOT / "aircheck" / "output"
BASE_MODEL_DIR = OUTPUT_DIR / "base_models" / BASE_MODEL.split("/")[-1]
OUT_DIR = OUTPUT_DIR / DATASET / "embeddings" / EMBEDDING_NAME

VIEW_KEYS = {"text": "TextModel", "image": "ImageModel", "graph": "Graph2dModel", "fused": "aggregator"}
unknown = set(VIEWS) - set(VIEW_KEYS)
assert not unknown, f"Unknown views {unknown}; choose from {list(VIEW_KEYS)}"

if MODEL_PATH is not None:
    MODEL_PATH = Path(MODEL_PATH).expanduser().resolve()
    if MODEL_PATH.is_dir():
        MODEL_PATH = MODEL_PATH / "best_model.ckpt"
    assert MODEL_PATH.exists(), f"Model not found: {MODEL_PATH}"
assert Path(DATA_PATH).exists(), f"Data not found: {DATA_PATH}"

config = dict(
    model=str(MODEL_PATH) if MODEL_PATH else f"base: {BASE_MODEL}",
    data_path=str(Path(DATA_PATH).resolve()), smiles_col=SMILES_COL,
    views=VIEWS, chunk_size=CHUNK_SIZE,
)
# Re-running with the same settings resumes; different settings need a new EMBEDDING_NAME
config_path = OUT_DIR / "config.json"
if config_path.exists():
    previous = json.loads(config_path.read_text())
    assert previous == config, (
        f"{OUT_DIR} holds an extraction with different settings:\n{previous}\n"
        "Choose a new EMBEDDING_NAME or delete that folder."
    )
    print("Resuming: finished chunks will be skipped.")
OUT_DIR.mkdir(parents=True, exist_ok=True)
config_path.write_text(json.dumps(config, indent=2))

print(f"Model  : {config['model']}")
print(f"Data   : {DATA_PATH}")
print(f"Views  : {VIEWS}")
print(f"Output : {OUT_DIR}")

Resuming: finished chunks will be skipped.
Model  : base: ibm-research/biomed.sm.mv-te-84m
Data   : ~/repos/biomed-multi-view/aircheck/sample_data/test.parquet
Views  : ['text', 'image', 'graph', 'fused']
Output : ~/repos/biomed-multi-view/aircheck/output/sample-data/embeddings/test-pretrained


## 2. Prepare data

Every SMILES is checked with RDKit first, because one unreadable SMILES would stop the whole run.
Invalid rows are skipped and listed in `invalid_smiles.csv`.

In [4]:
import pandas as pd
from rdkit import Chem, RDLogger

import pyarrow.parquet as pq


def read_table(path, columns, optional=()) -> pd.DataFrame:
    # Read only the needed columns from a .csv or .parquet file; optional columns are read if present
    path = Path(path)
    is_parquet = path.suffix.lower() in (".parquet", ".pq")
    available = pq.ParquetFile(path).schema_arrow.names if is_parquet else pd.read_csv(path, nrows=0).columns.tolist()
    missing = [c for c in columns if c not in available]
    assert not missing, f"{path.name} has no column(s) {missing}; available: {available}"
    wanted = list(dict.fromkeys([*columns, *(c for c in optional if c in available)]))
    df = pd.read_parquet(path, columns=wanted) if is_parquet else pd.read_csv(path, usecols=wanted)
    return df[wanted].reset_index(drop=True)

RDLogger.DisableLog("rdApp.*")

input_df = read_table(DATA_PATH, [SMILES_COL])

smiles = input_df[SMILES_COL].astype(str).str.strip()
valid = input_df[SMILES_COL].notna() & smiles.map(lambda s: Chem.MolFromSmiles(s) is not None)

molecules = pd.DataFrame({"row_index": input_df.index[valid], "smiles": smiles[valid]}).reset_index(drop=True)
if (~valid).any():
    input_df.loc[~valid, [SMILES_COL]].rename_axis("row_index").to_csv(OUT_DIR / "invalid_smiles.csv")

n_chunks = (len(molecules) + CHUNK_SIZE - 1) // CHUNK_SIZE
print(f"Rows           : {len(input_df):,}")
print(f"Valid SMILES   : {len(molecules):,} in {n_chunks:,} chunks of up to {CHUNK_SIZE:,}")
if (~valid).any():
    print(f"Invalid SMILES : {(~valid).sum():,} (skipped, listed in invalid_smiles.csv)")

Rows           : 1,031
Valid SMILES   : 1,031 in 2 chunks of up to 1,000


## 3. Load the model

In [5]:
import gc
import logging
import warnings

warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.WARNING)

import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from bmfm_sm.api.smmv_api import SmallMoleculeMultiViewModel
from bmfm_sm.core.data_modules.namespace import LateFusionStrategy, TaskType
from bmfm_sm.predictive.data_modules.multimodal_finetune_dataset import MultiModalFinetuneDataPipeline
from bmfm_sm.predictive.modules.finetune_lightning_module import FineTuneLightningModule

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(os.cpu_count())  # PyTorch may default to fewer threads than available CPUs

if MODEL_PATH is None:
    if not (BASE_MODEL_DIR / "model.safetensors").exists():
        from huggingface_hub import snapshot_download
        print(f"Downloading {BASE_MODEL} ...")
        snapshot_download(repo_id=BASE_MODEL, local_dir=BASE_MODEL_DIR)
    model = SmallMoleculeMultiViewModel.from_pretrained(
        LateFusionStrategy.ATTENTIONAL, model_path=str(BASE_MODEL_DIR), huggingface=True
    )
else:
    # The multi-view encoder inside the fine-tuned model (the prediction head is not needed)
    model = FineTuneLightningModule.load_from_checkpoint(MODEL_PATH, map_location="cpu").model
model.to(DEVICE).eval()
print(f"Model loaded on {DEVICE}")

2026-10-20 19:41:31,409 - root - INFO - instance-shay-ml-01:140646684186432:0:0 - Loading checkpoint via HuggingFace Hub from provided path ~/repos/biomed-multi-view/aircheck/output/base_models/biomed.sm.mv-te-84m


2026-10-20 19:41:32,728 - root - INFO - instance-shay-ml-01:140646684186432:0:0 - Using coeff_mlp architecture for aggregator


2026-10-20 19:41:32,729 - root - INFO - instance-shay-ml-01:140646684186432:0:0 - dim_list [512, 512, 768] for aggregator


Loading weights from local directory


2026-10-20 19:41:34,649 - root - INFO - instance-shay-ml-01:140646684186432:0:0 - in train False setting deterministic_eval = True


2026-10-20 19:41:35,165 - root - INFO - instance-shay-ml-01:140646684186432:0:0 - in train False setting deterministic_eval = True


Model loaded on cuda


## 4. Extract embeddings

Each chunk is converted into the three MMELON views (text tokens, 2D image, molecular graph),
encoded, and saved as one part file per view.

In [6]:
%%time
DATASET_ARGS = {
    "task_type": TaskType.CLASSIFICATION,
    "num_tasks": 1,
    "modalities": ["TEXT_MODEL", "IMAGE_MODEL", "GRAPH_2D_MODEL"],
    "smiles_col": "smiles",
    "label_cols": ["label"],
    "split_col": None,
}
pipeline_dir = OUT_DIR / "_pipeline"  # temporary input for the data pipeline, deleted at the end
pipeline_dir.mkdir(exist_ok=True)
for view in VIEWS:
    (OUT_DIR / view).mkdir(exist_ok=True)


def part_path(view: str, chunk: int) -> Path:
    return OUT_DIR / view / f"part-{chunk:05d}.parquet"


for chunk in tqdm(range(n_chunks), desc="chunks"):
    if all(part_path(v, chunk).exists() for v in VIEWS):
        continue  # finished in an earlier run
    chunk_df = molecules.iloc[chunk * CHUNK_SIZE:(chunk + 1) * CHUNK_SIZE]

    # The pipeline reads data_predict.csv and needs a label column (a placeholder here)
    chunk_df[["smiles"]].assign(label=0).to_csv(pipeline_dir / "data_predict.csv", index=False)
    dataset = MultiModalFinetuneDataPipeline(data_dir=str(pipeline_dir), dataset_args=DATASET_ARGS, stage="predict")
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                        collate_fn=dataset.collate_fn, num_workers=NUM_WORKERS)

    outputs = {view: [] for view in VIEWS}
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in batch.items()}
            separate = model.get_separate_outputs(batch)
            if "fused" in VIEWS:
                # The aggregator averages its view weights over the batch, so a batched fused embedding
                # would depend on which molecules share the batch. Fuse each molecule on its own instead.
                per_view = [separate[k] for k in ("Graph2dModel", "ImageModel", "TextModel")]  # aggregator order
                separate["aggregator"] = torch.cat(
                    [model.aggregator([v[i:i + 1] for v in per_view])[0] for i in range(len(per_view[0]))]
                )
            for view in VIEWS:
                outputs[view].append(separate[VIEW_KEYS[view]].float().cpu().numpy())

    for view in VIEWS:
        emb = np.concatenate(outputs[view])
        assert len(emb) == len(chunk_df), f"chunk {chunk}: {len(emb)} embeddings for {len(chunk_df)} molecules"
        part = chunk_df[["row_index", "smiles"]].assign(embedding=list(emb))
        # Write to a hidden temp file first, so an interrupted write never looks like a finished part
        final = part_path(view, chunk)
        tmp = final.with_name(f".{final.name}.tmp")
        part.to_parquet(tmp, index=False)
        tmp.rename(final)

    del dataset, loader, outputs
    gc.collect()

shutil.rmtree(pipeline_dir)
print(f"Done: {len(molecules):,} molecules, views {VIEWS}")

chunks:   0%|          | 0/2 [00:00<?, ?it/s]

2026-10-20 19:41:35,197 - root - INFO - instance-shay-ml-01:140646684186432:0:0 - Working on task TaskType.CLASSIFICATION


2026-10-20 19:41:35,197 - root - INFO - instance-shay-ml-01:140646684186432:0:0 - modalities running in data pipeline TEXT_MODEL,IMAGE_MODEL,GRAPH_2D_MODEL


2026-10-20 19:41:46,885 - root - INFO - instance-shay-ml-01:140646684186432:0:0 - Working on task TaskType.CLASSIFICATION


2026-10-20 19:41:46,885 - root - INFO - instance-shay-ml-01:140646684186432:0:0 - modalities running in data pipeline TEXT_MODEL,IMAGE_MODEL,GRAPH_2D_MODEL


Done: 1,031 molecules, views ['text', 'image', 'graph', 'fused']
CPU times: user 9.11 s, sys: 1.52 s, total: 10.6 s
Wall time: 14.2 s


## 5. Check and load the embeddings

Each view folder reads back as one table. `np.stack` turns the `embedding` column into a matrix.

In [7]:
for view in VIEWS:
    df_view = pd.read_parquet(OUT_DIR / view).sort_values("row_index").reset_index(drop=True)
    matrix = np.stack(df_view["embedding"].to_numpy())
    assert len(df_view) == len(molecules) and df_view["row_index"].is_unique
    print(f"{view:6s}: {matrix.shape[0]:,} molecules x {matrix.shape[1]} dims")

display(df_view.head())

text  : 1,031 molecules x 768 dims
image : 1,031 molecules x 512 dims
graph : 1,031 molecules x 512 dims
fused : 1,031 molecules x 512 dims


,row_index,smiles,embedding
0,0,Cc1ccc(C(=O)NC2CCN(C(=O)CCC(C)C)CC2)s1,"[0.3838811, 0.56576645, 0.51570475, 0.07523609..."
1,1,Cc1cccc(NS(=O)(=O)c2ccc(C(=O)N3CC(C)OC(C)C3)cc...,"[0.2470209, 0.21396814, 0.22440024, 0.00782004..."
2,2,N#Cc1cccc(S(=O)(=O)Nc2nnc(-c3ccccc3)o2)c1,"[0.1263078, 0.0, 0.059793882, 0.12992749, 0.24..."
3,3,COc1c(C)c2c(c(O)c1CC=C(C)CCC(=O)NCC(N)=O)C(=O)OC2,"[0.378241, 0.4575919, 0.4133663, 0.0422167, 0...."
4,4,O=C(CCc1nc(-c2ccsc2)no1)Nc1ccccc1N1CCOCC1,"[0.11773971, 0.0, 0.040200923, 0.086259924, 0...."


In [8]:
# Files produced
for path in sorted(OUT_DIR.rglob("*")):
    if path.is_file():
        print(f"{str(path.relative_to(OUT_DIR)):40s} {path.stat().st_size / 1e6:8.2f} MB")

config.json                                  0.00 MB
fused/part-00000.parquet                     2.12 MB
fused/part-00001.parquet                     0.07 MB
graph/part-00000.parquet                     2.70 MB
graph/part-00001.parquet                     0.10 MB
image/part-00000.parquet                     2.70 MB
image/part-00001.parquet                     0.10 MB
text/part-00000.parquet                      3.73 MB
text/part-00001.parquet                      0.14 MB
